In [4]:
from hyper_paramter import *

In [5]:
print(GLOBAL_DIMENSION)

1024


In [1]:
import os
os.environ["DASHSCOPE_API_KEY"] = "sk-eaa6c78b4d22459a8858b97d2dcac34e"
os.environ["DASHSCOPE_BASE_URL"] = "https://dashscope.aliyuncs.com/compatible-mode/v1"

In [2]:
from llama_index.llms.dashscope import DashScope, DashScopeGenerationModels

In [3]:

######这个接口支持的模型比较少,这样调用没反应
#####先用turbo顶一下
def get_dashscope_qwen_turbo():
    dashscope_llm = DashScope(model_name=DashScopeGenerationModels.QWEN_TURBO,
                          base_url ="https://dashscope.aliyuncs.com/compatible-mode/v1",
                          max_tokens=798,
                          temperature=1.0,
                          api_key=os.getenv("DASHSCOPE_API_KEY"))
    return dashscope_llm

In [4]:
# from llama_index.llms.openai import OpenAI
# from llama_index.embeddings.openai import OpenAIEmbedding
# from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.node_parser import SentenceWindowNodeParser
from llama_index.core.node_parser import SentenceSplitter

# create the sentence window node parser w/ default settings
node_parser = SentenceWindowNodeParser.from_defaults(
    window_size=3,
    window_metadata_key="window",
    original_text_metadata_key="original_text",
)

In [5]:
llm = get_dashscope_qwen_turbo()

In [7]:
from langchain_community.embeddings.dashscope import DashScopeEmbeddings
def get_text_embeddings():
    '''目前使用的llama-index的方法，会让系统报错，为什么呢，默认为1536？
    这个langchain_community默认嵌入维度为1024'''
    embeddings = DashScopeEmbeddings(
    model="text-embedding-v3",
    dashscope_api_key=os.getenv("DASHSCOPE_API_KEY")
)
    return embeddings

In [8]:
embed_model = get_text_embeddings()

In [9]:
# base node parser is a sentence splitter
text_splitter = SentenceSplitter()

from llama_index.core import Settings

Settings.llm = llm
Settings.embed_model = embed_model
Settings.text_splitter = text_splitter

In [5]:
from llama_index.core import SimpleDirectoryReader

documents = SimpleDirectoryReader(
    input_files=["/mnt/chatchat/aimtoMD/AILAB工具文件_实验.pdf"]
).load_data()

In [7]:
type(documents[0]),len(documents)

(llama_index.core.schema.Document, 15)

In [8]:
nodes = node_parser.get_nodes_from_documents(documents)

In [9]:
len(nodes)

147

In [15]:
base_nodes = text_splitter.get_nodes_from_documents(documents)

In [16]:
from llama_index.core import VectorStoreIndex

sentence_index = VectorStoreIndex(nodes)

In [17]:
base_index = VectorStoreIndex(base_nodes)

In [ ]:
from llama_index.core.postprocessor import MetadataReplacementPostProcessor

query_engine = sentence_index.as_query_engine(
    similarity_top_k=5,
    # the target key defaults to `window` to match the node_parser's default
    node_postprocessors=[
        MetadataReplacementPostProcessor(target_metadata_key="window")
    ],
)
window_response = query_engine.query(
    "关于支持向量机模型，非参数优化，随机搜索，有什么检索到的信息？"
)
print(window_response) ###好的，屁都没搜出来

对于支持向量机（SVM）模型，在用户选择非参数优化的情况下，相关的信息包括：

7. 正则化参数 C  
默认值是 1， 用户可以选择 0.0001, 0.001, 0.01, 0.1, 1, 10, 100，也可以自定义取值范围是 0.0001 到 100 的小数点后 5 位  
提示语：正则化参数，控制软间隔大小。C 越大，模型越偏向减少训练误差（可能过拟合）；越小，越偏向于泛化。

8. 核函数 kernel  
默认是高斯径向函数， 用户可以选择线性核 (linear)， 多项式核(poly)，高斯径向函数(rbf)  
提示语： 指定核函数类型，决定了特征如何映射到高维空间。

9. 高斯径向函数参数 g  
默认是 scale, 用户可以选择 scale， auto, 或是自定义 0.0001-100 的小数点后 5 位  

如果用户选择的参数优化方法为随机搜索，则页面如下：
15. 最小 C 值 (C_min)  
默认值为 0.0001， 范围是 0.0001 到 100 的小数点后 5 位，用户可自定义此参数  

16. 最大 C 值 (C_max)  
默认值为 100， 范围是 0.0001 到 100 的小数点后 5 位，用户可自定义此参数  

17. 最小核函数参数 g (g_min)  
默认值为 0.0001， 范围是 0.0001 到 100 的小数点后 5 位，用户可自定义此参数  

18. 最大核函数参数 g (g_max)  
默认值为 100， 范围是 0.0001 到 100 的小数点后 5 位，用户可自定义此参数  

这些信息提供了关于 SVM 模型中正则化参数 C、核函数类型及其参数 g 的默认值、可选值以及用户自定义的范围和提示语。


We can also check the original sentence that was retrieved for each node, as well as the actual window of sentences that was sent to the LLM.

In [19]:
window = window_response.source_nodes[0].node.metadata["window"]
sentence = window_response.source_nodes[0].node.metadata["original_text"]

print(f"Window: {window}")
print("------------------")
print(f"Original Sentence: {sentence}")

Window: 提示语：交叉验证中的折数，表示将训练集划分为几等份轮流做训练和验证，以更稳定地评估
模型性能 
3.  交叉验证方法（cv_method):  
默认是为 KFold, 从 KFold, Stratified kfold, leave_one_out 这三个选项选择 
如果用户选择 leave_one_out 的选项，就把 cv 隐掉，不能选择cv 
提示语：将数据集划分为多个子集，轮流用不同子集进行训练和测试 
4.  选择特征数量 （top_n)  
默认是 4， 用户可以自定义，要求正整数，2-50 
提示语：选择特征数量绘制特征重要性图和 shap  
选择特征数量的界面：Slider with input boxSlider | Element Plus 
5.  参数优化方法：(search_method) 
默认是不进行超参数优化，用户可以选择不进行超参数优化，随机搜索，贝叶斯优化，网格搜
索，此为单选 
提示语： 参数优化方法是通过系统地搜索不同超参数组合，以找到能使模型性能最优的配
置。 
----------------------------------------------------------------------------------
----- 
如果用户选择的参数优化方法为不进行超参数优化，则页面如下：  
这里在 json 文件中，是[optimization_config][RandomForest][no_optimization] 
1.  树的数量（n_estimators ） 
默认值为 100， 范围 10-1000 正整数，用户可自定义此参数 
提示语：表示你要在森林中生成多少棵决策树 
2.  树的最大深度(max_depth) 
默认值为：10， 提供选项 1，3，5， 10， 用户也可以选择自定义参 
数，范围是 1-50 正整数 
提示语：它控制 每棵决策树可以生长的最大层数，决定了模型对训练数据的拟合能力 
3.  每次分裂所考虑的最大特征数（max_features) 
默认值为：sqrt, 提供选项，‘log2', 'None', 'sqrt'  
提示语：控制 每次寻找最佳分裂时随机选取的特征子集大小 
4. 
------------------
Original Senten

In [ ]:
query_engine = base_index.as_query_engine(similarity_top_k=5)
vector_response = query_engine.query(
    "关于支持向量机模型，非参数优化，随机搜索，有什么检索到的信息？"
)
print(vector_response)
###它是检索到的信息都集中在随机森林模型中

对于支持向量机模型，在非参数优化的情况下，没有相关的检索信息。所有提供的信息都集中在随机森林模型的相关参数设置和优化方法上，包括树的数量、最大深度、最小样本分裂数、叶节点最小样本数、特征选择数量以及交叉验证方法等。这些信息并未涉及支持向量机模型的具体参数或优化策略。


In [21]:
query_engine = base_index.as_query_engine(similarity_top_k=10)
vector_response = query_engine.query(
    "关于支持向量机模型，非参数优化，随机搜索，有什么检索到的信息？"
)
print(vector_response)

关于支持向量机模型，在非参数优化的情况下，随机搜索的相关信息未直接检索到。然而，对于支持向量机的参数优化方法，提供了多种选择，包括不进行超参数优化、随机搜索、贝叶斯优化和网格搜索。在非参数优化的场景下，支持向量机通常会使用默认参数进行模型训练。具体提到的默认参数包括正则化参数 \(C\)、核函数类型 \(kernel\) 和高斯径向函数参数 \(gamma\) 等。这些参数的具体范围和默认值可以根据需要进行调整，以适应不同的应用场景。


In [22]:
for node in vector_response.source_nodes:
    print("SVM; 非参数优化, 随机搜索 mentioned?", "随机搜索" in node.node.text)
    print("--------")

SVM; 非参数优化, 随机搜索 mentioned? True
--------
SVM; 非参数优化, 随机搜索 mentioned? False
--------
SVM; 非参数优化, 随机搜索 mentioned? True
--------
SVM; 非参数优化, 随机搜索 mentioned? False
--------
SVM; 非参数优化, 随机搜索 mentioned? True
--------
SVM; 非参数优化, 随机搜索 mentioned? True
--------
SVM; 非参数优化, 随机搜索 mentioned? False
--------
SVM; 非参数优化, 随机搜索 mentioned? True
--------
SVM; 非参数优化, 随机搜索 mentioned? False
--------
SVM; 非参数优化, 随机搜索 mentioned? True
--------


LLM读不到中间文本的辅助信息，即使已经检索到相关信息

参考这里做metric

https://docs.llamaindex.ai/en/stable/examples/node_postprocessor/MetadataReplacementDemo/

## 层次NodeParser

In [ ]:
'''This node parser will chunk nodes into hierarchical nodes. 
This means a single input will be chunked into several hierarchies of chunk sizes, 
with each node containing a reference to it's parent node.

When combined with the AutoMergingRetriever, 
this enables us to automatically replace retrieved nodes with their parents when a majority of children are retrieved. 
This process provides the LLM with more complete context for response synthesis.

https://docs.llamaindex.ai/en/stable/examples/retrievers/auto_merging_retriever/
'''


In [ ]:
from llama_index.core.node_parser import HierarchicalNodeParser

node_parser = HierarchicalNodeParser.from_defaults(
    chunk_sizes=[2048, 512, 128]
)

## semantic chunk语义分块

In [23]:
from llama_index.core.node_parser import SemanticSplitterNodeParser
# from llama_index.embeddings.openai import OpenAIEmbedding

In [25]:
import os
from typing import Any, List
import asyncio
from llama_index.core.embeddings import BaseEmbedding
from llama_index.core.bridge.pydantic import PrivateAttr
import dashscope
from http import HTTPStatus

class CustomDashScopeEmbedding(BaseEmbedding):
    """
    同样的，还有一个CustomLLM
    自定义 DashScope 嵌入模型，继承自 BaseEmbedding。
    封装 dashscope.TextEmbedding.call 的逻辑，使其适配 LlamaIndex 框架。
    最终实现了对向量数据库dim参数的支持，否则只能使用模型的默认输出
    """

    _model_name: str = PrivateAttr()
    _api_key: str = PrivateAttr()
    _base_url: str = PrivateAttr()
    _dimension: int = PrivateAttr()
    _output_type: str = PrivateAttr()

    def __init__(
        self,
        model_name: str = "text-embedding-v4",
        api_key: str = None,
        base_url: str = None,
        dimension: int = None,
        output_type: str = "dense",
        **kwargs: Any,
    ) -> None:
        super().__init__(**kwargs)
        self._model_name = model_name
        self._api_key = api_key if api_key else os.getenv("DASHSCOPE_API_KEY")
        self._base_url = base_url if base_url else os.getenv("DASHSCOPE_BASE_URL")
        self._dimension = dimension
        self._output_type = output_type

        if not self._api_key:
            raise ValueError(
                "DashScope API Key is not provided. "
                "Please set it via api_key argument or DASHSCOPE_API_KEY environment variable."
            )

        dashscope.api_key = self._api_key
        if self._base_url:
            dashscope.base_url = self._base_url

    @classmethod
    def class_name(cls) -> str:
        return "custom_dashscope_embedding"

    # --- 同步方法 ---
    def _get_query_embedding(self, query: str) -> List[float]:
        return self._get_text_embedding_batch([query])[0]

    def _get_text_embedding(self, text: str) -> List[float]:
        return self._get_text_embedding_batch([text])[0]

    def _get_text_embedding_batch(self, texts: List[str]) -> List[List[float]]:
        if not texts:
            return []

        call_params = {
            "model": self._model_name,
            "input": texts,
            "output_type": self._output_type,
        }
        if self._dimension:
            call_params["dimension"] = self._dimension

        try:
            resp = dashscope.TextEmbedding.call(**call_params)

            if resp.status_code == HTTPStatus.OK:
                embeddings = []
                for entry in resp.output['embeddings']:
                    embeddings.append(entry['embedding'])
                return embeddings
            else:
                # If DashScope API returns an error status, raise an exception
                raise Exception(
                    f"DashScope API 调用失败。Status Code: {resp.status_code}, Message: {resp.message}"
                )
        except Exception as e:
            # Re-raise the exception to indicate a failure in getting embeddings
            raise RuntimeError(f"调用 DashScope 嵌入 API 时发生错误: {e}") from e

    # --- 异步方法 ---
    async def _aget_query_embedding(self, query: str) -> List[float]:
        return (await self._aget_text_embedding_batch([query]))[0]

    async def _aget_text_embedding(self, text: str) -> List[float]:
        return (await self._aget_text_embedding_batch([text]))[0]

    async def _aget_text_embedding_batch(self, texts: List[str]) -> List[List[float]]:
        if not texts:
            return []

        call_params = {
            "model": self._model_name,
            "input": texts,
            "output_type": self._output_type,
        }
        if self._dimension:
            call_params["dimension"] = self._dimension

        loop = asyncio.get_event_loop()
        try:
            resp = await loop.run_in_executor(
                None,
                lambda: dashscope.TextEmbedding.call(**call_params)
            )

            if resp.status_code == HTTPStatus.OK:
                embeddings = []
                for entry in resp.output['embeddings']:
                    embeddings.append(entry['embedding'])
                return embeddings
            else:
                # If DashScope API returns an error status, raise an exception
                raise Exception(
                    f"DashScope API 调用失败。Status Code: {resp.status_code}, Message: {resp.message}"
                )
        except Exception as e:
            # Re-raise the exception to indicate a failure in getting embeddings
            raise RuntimeError(f"调用 DashScope 嵌入 API 时发生错误: {e}") from e

In [ ]:
# from llm.embedding import CustomDashScopeEmbedding###使用自定义的EMBEDDING类，与llama-index和langchain的集成不同，可以设置
def get_embed_model(dim: int=1024):
    embedding = CustomDashScopeEmbedding(
            model_name="text-embedding-v4", ##支持的维度64->2048; 1536;
            dimension=dim, # 例如，如果您的模型支持且需要指定维度
            output_type="dense"
        )
    ## output_type= [dense、sparse、dense&sparse]
    # https://help.aliyun.com/zh/model-studio/text-embedding-synchronous-api?spm=a2c4g.11186623.help-menu-2400256.d_2_6_0.7cb548234p7QgN
    return embedding

In [27]:
embed_model = get_embed_model()

In [28]:
Settings.embed_model = embed_model

In [29]:
splitter = SemanticSplitterNodeParser(
    buffer_size=1, breakpoint_percentile_threshold=95, embed_model=embed_model
)

https://docs.llamaindex.ai/en/stable/examples/node_parsers/semantic_chunking/

In [30]:
nodes = splitter.get_nodes_from_documents(documents)

In [34]:
len(nodes)

30

In [31]:
print(nodes[1].get_content())

高斯径向函数参数 g 
默认是 scale, 用户可以选择 scale， auto, 或是自定义0.0001-100 的小数点后 5 位 


In [32]:
print(nodes[2].get_content())

如果用户选择的核函数为线性核，则不选这个参数 
提示语 ：控制 rbf, poly, sigmoid 核函数的影响范围 
10. 多项式核参数 d  
默认是 3， 只有当核函数选为多项式核的时候才有效，用户可以选 2，3，4 
提示语： 表示多项式的次数。它控制了输入特征被映射到高维空间的方式，对模型复杂度影
响较大。 
11. 类别权重 class_weight 
默认是 balanced, 用户可以选择 None 或是 balanced 
提示语： 用于处理类别不平衡。 
----------------------------------------------------------------------------------
----- 
如果用户选择的参数优化方法为随机搜索，则页面如下：  
12. 


In [33]:
print(nodes[3].get_content())

目标函数 scoring 
默认是 accuracy, 用户可以选择 precision, accuracy, recall  
单选 
13. 随机搜索次数 n_iter 
默认是 20， 用户可以选择 2- 50 正整数 
----------------------------------------------------------------------------------
----- 
如果用户选择的参数优化方法为网格搜索，则页面如下：  
14. 目标函数 scoring 
默认是 accuracy, 用户可以选择 precision, accuracy, recall  
单选 
----------------------------------------------------------------------------------
----- 
如果用户选择的参数优化方法为贝叶斯优化，则页面如下：  
15. 目标函数 scoring 
默认是 accuracy, 用户可以选择 precision, accuracy, recall  
单选 
16. 贝叶斯搜索次数 n_iter 
默认是 20， 用户可以选择 2- 50 正整数 
 
回归任务： 
回归任务没有随机种子(random_state)和 class_weight 这个参数 
其余参数一样，除了： 
1. 目标函数 scoring 
默认是 neg_mean_squared_error, 用户可以选择 neg_mean_squared_error, 
neg_root_mean_squared_error, r2，neg_mean_absolute_error 这个是单选 
命令： 


### Query Engine

In [36]:
#
from llama_index.core import VectorStoreIndex
from llama_index.core.response.notebook_utils import display_source_node
vector_index = VectorStoreIndex(nodes)
query_engine = vector_index.as_query_engine()

In [37]:
response = query_engine.query(
    "请告诉我关于支持向量机（SVM）的一切！"
)

In [38]:
print(str(response))

支持向量机（SVM）是一种广泛应用于分类和回归任务的机器学习算法。以下是关于SVM的一些关键信息：

1. **分类任务**：
   - **公共参数**：
     - **测试集比例(test_size)**：可以选择的比例包括0.1到0.9，默认值为0.2。
     - **随机种子(random_state)**：默认值为42，范围为0-1000的随机正整数。
     - **交叉验证数(cv)**：默认值为5，范围是3-20的正整数。
     - **交叉验证方法(cv_method)**：默认为KFold，可选KFold、StratifiedKFold、LeaveOneOut。
     - **选择特征数量(top_n)**：默认值为4，用户可以自定义。
     - **参数优化方法(search_method)**：默认为不进行优化，用户可以选择不优化、随机搜索、贝叶斯优化或网格搜索。

2. **回归任务**：
   - **分类纯度方法(criterion)**：默认值为squared_error，用户可以选择squared_error或absolute_error。

3. **参数优化**：
   - 如果选择不进行超参数优化，则需要手动设置以下参数：
     - **正则化参数c**：默认值为1，可选值包括0.0001到100的小数点后5位。
     - **核函数kernel**：默认为高斯径向函数，可选线性核、多项式核和高斯径向函数。

4. **结果评估**：
   - **模型评估指标**：包括MAE（平均绝对误差）、MSE（均方误差）、RMSE（均方根误差）、R²（决定系数）、MAPE（平均绝对百分比误差）。
   - **图形输出**：包括残差图、残差分布图、预测值与实际值图、特征重要性图和SHAP图。

5. **其他功能**：
   - **随机森林**：在某些情况下可能会使用随机森林作为对比模型。
   - **界面展示**：提供用户友好的界面来上传数据和设置参数。

这些信息涵盖了SVM的基本原理、常用参数及其应用方法。


In [51]:
response = query_engine.query(
    "辅助文档里面有关于支持向量机的信息大概讲了什么？请给我分点列出"
)
print(str(response))

1. 支持向量机既可用于分类任务，也可用于回归任务。
2. 在分类任务中，提到分类纯度方法，用户可以选择不同的方式来衡量分类纯度。
3. 对于测试集比例，用户可以选择从 0.1 到 0.9 的不同值，默认值为 0.2，用于控制测试集占整个数据集的比例。
4. 随机种子默认值为 42，范围在 0-1000 之间，用于确保数据集划分方式一致，使结果可复现。
5. 交叉验证数默认值为 5，范围是 3-20 的正整数，用户可以自定义。
6. 交叉验证方法有 KFold、StratifiedKFold 和 LeaveOneOut 三种，默认为 KFold。
7. 用户可以选择特征数量绘制特征重要性图和 SHAP 图，默认值为 4。
8. 参数优化方法包括不进行优化、随机搜索、贝叶斯优化和网格搜索。
9. 如果不进行超参数优化，用户需要设置正则化参数 C 和核函数类型，其中 C 默认值为 1，核函数默认为高斯径向函数。


In [42]:
def pretty_print(text, words_per_line=50):
    # 将文本按空格分割为单词
    words = text.split()
    
    # 每 50 个单词组成一行
    lines = []
    for i in range(0, len(words), words_per_line):
        lines.append(' '.join(words[i:i + words_per_line]))
    
    # 将所有行用换行符连接起来
    formatted_text = '\n'.join(lines)
    print(formatted_text)

In [45]:
type(response)

llama_index.core.base.response.schema.Response

In [49]:
response.response

'辅助文档里面关于支持向量机的信息主要涵盖了公共参数和一些具体设置。公共参数包括测试集比例、随机种子、交叉验证数、交叉验证方法、选择特征数量以及参数优化方法。此外，还详细说明了分类任务和回归任务的相关提示语和默认值，以及如何选择不同的核函数和正则化参数等。文档还提到了界面展示和示例数据的上传过程。'

In [50]:
pretty_print(str(response.response),words_per_line=10)

辅助文档里面关于支持向量机的信息主要涵盖了公共参数和一些具体设置。公共参数包括测试集比例、随机种子、交叉验证数、交叉验证方法、选择特征数量以及参数优化方法。此外，还详细说明了分类任务和回归任务的相关提示语和默认值，以及如何选择不同的核函数和正则化参数等。文档还提到了界面展示和示例数据的上传过程。


In [52]:
for n in response.source_nodes:
    display_source_node(n, source_length=20000)

**Node ID:** c06143ca-70d1-45bc-a09c-1ba086630f70<br>**Similarity:** 0.6205676028742384<br>**Text:** singularity exec --bind /ailab/svm,/prog1,/ailab /prog1/Container/AILab/ailab_v03.sif python 
/ailab/svm/test/2025/Script/0_SVM_r.py -c /ailab/svm/test/2025/AI_SVM_parameters.json 
结果预览：  
预览 7 个文件 
model_metrics  
MAE: 平均绝对误差 
MSE: 均方误差 
RMSE: 均方根误差 
R
2
: 决定系数 
MAPE: 平均绝对百分比误差 
Testing score: 测试集分数 
Cross-validation score: 交叉验证分数 
residuals_vs_fitted.png 残差图 
error_distribution.png 残差分布图 
Prediction_vs_Actual.png 预测值与实际值图 
Feature_importance_plot.png 
Shap_plot.png 
参考资料 
支持向量机分类任务 
支持向量机回归任务 
随机森林：highlight 是页面更新的部分 
界面展示 ： 
参照 SVM 的界面：  
示例数据 
上传数据 
回归任务 
1. 分类纯度方法(criterion) -cr 
默认值是 squared_error, 用户可以选择 squared_error, absolute_error 
提示语：都是衡量 分类纯度 的方式 
分类任务 
1. 测试集比例：test_size,  
可以选择，0.1， 0.2， 0.3，0.4，0.5，0.6，0.7，0.8，0.9 （默认值为 0.2） 
提示语：控制测试集占整个数据集的比例 
1.<br>

**Node ID:** 87b26ab6-6ba8-4183-b65d-d91dfa3be659<br>**Similarity:** 0.5914845304207672<br>**Text:** SVM 
分类任务： highlight 是页面更新的部分 
公共参数：  
1. 测试集比例：(test_size), 
可以选择，0.1， 0.2， 0.3，0.4，0.5，0.6，0.7，0.8，0.9 （默认值为 0.2） 
提示语：控制测试集占整个数据集的比例 
2. 随机种子(random_state) 
默认值为 42，范围 0-1000，随机正整数 
提示语：随机种子未来确保数据集的划分方式一致，结果可复现 
3. 交叉验证数(cv) 
默认值为 5， 范围是 3-20 正整数，用户自定义 
提示语：交叉验证中的折数，表示将训练集划分为几等份轮流做训练和验证，以更稳定地评估
模型性能 
4. 交叉验证方法（cv_method): 
默认是为 KFold, 从 KFold, StratifiedKFold, LeaveOneOut 这三个选项选择 
如果用户选择 leave_one_out 的选项，就把 cv 隐掉，不能选择cv 
提示语：将数据集划分为多个子集，轮流用不同子集进行训练和测试 
5. 选择特征数量 （top_n)  
默认是 4， 用户可以自定义，要求正整数，2-无限制 
提示语：选择特征数量绘制特征重要性图和 shap  
选择特征数量的界面：Slider with input boxSlider | Element Plus 
6. 参数优化方法：(search_method) 
默认是不进行超参数优化，用户可以选择不进行超参数优化，随机搜索，贝叶斯优化，网格搜
索，此为单选 
提示语： 参数优化方法是通过系统地搜索不同超参数组合，以找到能使模型性能最优的配
置。 
----------------------------------------------------------------------------------
----- 
如果用户选择的参数优化方法为不进行超参数优化，则页面如下：  
7. 正则化参数 c 
默认是 1， 用户可以选择 0.0001, 0.001, 0.01, 0.1, 1, 10, 100 也可以自定义取值范围是 0.0001 - 
100 的小数点后 5 位 
提示语：正则化参数，控制软间隔大小。C 越大，模型越偏向减少训练误差（可能过拟合）；
越小，越偏向于泛化。 
8. 核函数 kernel 
默认是高斯径向函数， 用户可以选择线性核 (linear)， 多项式核(poly)，高斯径向函数(rbf)  
提示语： 指定核函数类型，决定了特征如何映射到高维空间。 
9.<br>